In [ ]:
import os
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
thresholds = ['0.1','0.5','0.8','A','I','N']
hardware = ["kyiv", "brisbane", "sherbrooke"]
mutant_types = ["equiv", "normal", "balanced"]
metrics = ['C', 'H', 'J', 'T', 'F', 'E']
metric_names=['chisquare', 'hellinger', 'jensenshannon', 'trace', 'fidelity', 'expectation']
output_type = {'ae': 'Dominant', 'qpeexact': 'Dominant', 'vqe': 'Dominant', 'qft': 'Diverse', 'qftentangled': 'Diverse', 'wstate': 'Diverse'}


In [ ]:
def getModelTolerance(model):
    if model == 'brisbane':
        tolerance_values_noisy = {
            'fidelity': 1 - 0.9818071588272935,
            'trace': 0.9474790361650993,
            'hellinger': 0.9001136659697,
            'jensenshannon': 0.7718372511093367,
            'chisquare': 8.822528185214266e-158,
            'expectation': 0.7070786758337857
        }
    elif model == 'sherbrooke':
        tolerance_values_noisy = {
            'fidelity': 1 - 0.9817918801809562,
            'trace': 0.9165467778554671,
            'hellinger': 0.8723835633308122,
            'jensenshannon': 0.7525100350370049,
            'chisquare': 3.1954543753995917e-141,
            'expectation': 0.5896550492107876
        }
    elif model == 'kyiv':
        tolerance_values_noisy = {
            'fidelity': 1 - 0.9817973573897236,
            'trace': 0.9134759188590066,
            'hellinger': 0.8760278316636461,
            'jensenshannon': 0.7549426398105366,
            'chisquare': 5.687357104528032e-162,
            'expectation': 0.6140893479852809
        }

    else:
        tolerance_values_noisy = {}

    return tolerance_values_noisy

def cap_value(value):
    return min(1, max(0, value))

def get_tolerance_values_noisy(model, threshold):
    tolerance_values_ideal = {
        'fidelity': 1 - 1e-14,
        'trace': 1e-13,
        'hellinger': 0.13455009062719828,
        'jensenshannon': 0.11716009455796059,
        'chisquare': 0.318714816155845,
        'expectation': 0
    }
    # Define tolerance values
    if threshold == 'I':
        tolerance_values_noisy = tolerance_values_ideal
    elif threshold == 'N':
        tolerance_values_noisy = getModelTolerance(model)
    elif threshold == 'A':
        tolerance_values_noisy = getModelTolerance(model)
        tolerance_values_noisy = {
            'fidelity':  cap_value(1 - ((1 - tolerance_values_noisy['fidelity']) + (1 - tolerance_values_ideal['fidelity']))),
            'trace': cap_value(tolerance_values_noisy['trace'] + tolerance_values_ideal['trace']),
            'hellinger': cap_value(tolerance_values_noisy['hellinger'] + tolerance_values_ideal['hellinger']),
            'jensenshannon': cap_value(tolerance_values_noisy['jensenshannon'] + tolerance_values_ideal['jensenshannon']),
            'chisquare': cap_value(tolerance_values_noisy['chisquare'] + tolerance_values_ideal['chisquare']),
            'expectation': cap_value(tolerance_values_noisy['expectation'] + tolerance_values_ideal['expectation'])
        }

    else:
        tolerance_values_noisy = {
            'fidelity': 1 - threshold,
            'trace': threshold,
            'hellinger': threshold,
            'jensenshannon': threshold,
            'chisquare': threshold,
            'expectation': threshold
        }

    return tolerance_values_noisy


In [ ]:
# Helper function to setup layout and save image
def setup_layout_and_save(fig, title, folder_name, file_name, yaxis_range=None):
    fig.update_layout(
        title_text=title,
        height=400,
        width=2000,
        showlegend=True,
        yaxis_range=yaxis_range  # Set y-axis range if provided
    )
    os.makedirs(folder_name, exist_ok=True)
    fig.write_image(f"{folder_name}/{file_name}.png")# engine='orca')

In [ ]:
def get_grouped_bar_chart(noise_model):
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold
    
    tolerance_values_ideal = get_tolerance_values_noisy(noise_model, "I")
    
    # Add bars for each category
    for i, threshold in enumerate([0.1,0.5,0.8,'A','I','N']):
        
        tolerance_values_noisy = get_tolerance_values_noisy(noise_model, threshold)
        y_values = [tolerance_values_noisy[metric] for metric in metric_names]
        fig.add_trace(go.Bar(
            name=threshold,  # mutant type
            x=metric_names,  # metric_thresholds
            y=y_values,  # F1 scores
            hoverinfo='y',
            marker=dict(color=color_scale[i % len(color_scale)])
        ))

    # Add horizontal lines for the ideal tolerance values
    for metric_index, metric in enumerate(metric_names):
        ideal_value = tolerance_values_ideal.get(metric)
        fig.add_shape(
            type="line",
            x0=metric_index - 0.5,  # Start of the group
            x1=metric_index + 0.5,  # End of the group
            y0=ideal_value, y1=ideal_value, xref='x',
            yref='y',
            line=dict(color="red", width=3)
        )            

    return fig

In [ ]:
for hw in hardware:
    fig = get_grouped_bar_chart(hw)
    setup_layout_and_save(fig, f"Thresholds comparison for {hw} noise model", f"results", f"thresholds_{hw}")
    fig.show()